In [ ]:
import argparse
import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd


def find_excel_files(root_dir: Path) -> List[Path]:
    exts = {".xlsx", ".xlsm", ".xls"}  # .xls may fail without xlrd; handled gracefully
    files = []
    for p in root_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in exts and not p.name.startswith("~$"):
            files.append(p)
    return sorted(files)


def _drop_empty_edges(df: pd.DataFrame) -> pd.DataFrame:
    """Trim fully-empty rows/cols (like Excel used range)."""
    if df.empty:
        return df
    df2 = df.copy()
    df2 = df2.dropna(axis=0, how="all").dropna(axis=1, how="all")
    return df2


def _make_unique_columns(cols: List[str]) -> List[str]:
    seen: Dict[str, int] = {}
    out = []
    for c in cols:
        base = c
        if base in seen:
            seen[base] += 1
            out.append(f"{base}__{seen[base]}")
        else:
            seen[base] = 0
            out.append(base)
    return out


def normalize_binary_value(x):
    """Normalize typical binary encodings to {0,1} when possible; else return original."""
    if pd.isna(x):
        return np.nan
    # bools
    if isinstance(x, (bool, np.bool_)):
        return 1 if bool(x) else 0
    # numerics
    if isinstance(x, (int, np.integer)) and x in (0, 1):
        return int(x)
    if isinstance(x, (float, np.floating)) and np.isfinite(x) and x in (0.0, 1.0):
        return int(x)
    # strings
    if isinstance(x, str):
        s = x.strip().lower()
        if s in {"0", "0.0", "false", "f", "no", "n"}:
            return 0
        if s in {"1", "1.0", "true", "t", "yes", "y"}:
            return 1
    return x


def is_binary_series(s: pd.Series) -> Tuple[bool, Optional[pd.Series]]:
    """Return (is_binary, normalized_series_if_binary_else_None)."""
    if s is None:
        return False, None
    sn = s.map(normalize_binary_value)
    sn_nonnull = sn.dropna()
    if sn_nonnull.empty:
        return False, None
    uniq = set(pd.unique(sn_nonnull))
    if uniq.issubset({0, 1}):
        return True, sn
    return False, None


def sanitize_sheet_name(name: str) -> str:
    return re.sub(r"[\x00-\x1f]", "", name)[:31]


def analyze_sheet(file_path: Path, sheet_name: str) -> Tuple[Optional[dict], List[dict]]:
    """
    Returns:
      - table_summary (dict) or None if sheet can't be processed / empty
      - variables_summaries (list of dicts)
    """
    try:
        raw = pd.read_excel(
            file_path,
            sheet_name=sheet_name,
            header=None,
            engine="openpyxl" if file_path.suffix.lower() in {".xlsx", ".xlsm"} else None,
        )
    except Exception as e:
        # .xls often needs xlrd; treat as skip with error recorded upstream
        return None, [{"file": str(file_path), "sheet": sheet_name, "error": str(e)}]

    raw = _drop_empty_edges(raw)
    if raw.empty or raw.shape[0] < 1 or raw.shape[1] < 1:
        return None, []

    # Assume first row is header
    header_row = raw.iloc[0].astype(str).fillna("").tolist()
    header_row = [h.strip() if isinstance(h, str) else str(h) for h in header_row]
    header_row = [h if h and h.lower() != "nan" else f"unnamed_{i}" for i, h in enumerate(header_row)]
    header_row = _make_unique_columns(header_row)

    data = raw.iloc[1:].copy()
    data.columns = header_row

    # If after removing header, no data rows remain, still report "table" size with 0 rows
    n_rows = int(data.shape[0])
    n_cols = int(data.shape[1])

    # Table-level summary
    table_summary = {
        "file": str(file_path),
        "sheet": sheet_name,
        "rows": n_rows,
        "cols": n_cols,
        "columns": ", ".join(map(str, header_row)),
    }

    # Variable-level summaries
    vars_out: List[dict] = []
    for col in data.columns:
        s = data[col]
        non_missing = int(s.notna().sum())
        missing = int(s.isna().sum())
        # Unique excluding NaN
        try:
            n_unique = int(s.dropna().nunique())
        except Exception:
            n_unique = None

        var_rec = {
            "file": str(file_path),
            "sheet": sheet_name,
            "variable": str(col),
            "non_missing": non_missing,
            "missing": missing,
            "n_unique": n_unique,
            "is_binary": False,
            "binary_0_count": None,
            "binary_1_count": None,
        }

        is_bin, sn = is_binary_series(s)
        if is_bin and sn is not None:
            sn_nonnull = sn.dropna().astype(int)
            var_rec["is_binary"] = True
            var_rec["binary_0_count"] = int((sn_nonnull == 0).sum())
            var_rec["binary_1_count"] = int((sn_nonnull == 1).sum())

        vars_out.append(var_rec)

    return table_summary, vars_out


def analyze_excel_file(file_path: Path) -> Tuple[List[dict], List[dict], List[dict]]:
    """
    Returns:
      - table_summaries
      - variable_summaries
      - errors
    """
    tables: List[dict] = []
    vars_all: List[dict] = []
    errors: List[dict] = []

    try:
        xls = pd.ExcelFile(
            file_path,
            engine="openpyxl" if file_path.suffix.lower() in {".xlsx", ".xlsm"} else None,
        )
        sheet_names = xls.sheet_names
    except Exception as e:
        errors.append({"file": str(file_path), "sheet": None, "error": f"Failed to open ExcelFile: {e}"})
        return tables, vars_all, errors

    for sh in sheet_names:
        table_summary, vars_out = analyze_sheet(file_path, sh)
        if table_summary is not None:
            tables.append(table_summary)
            vars_all.extend(vars_out)
        else:
            # vars_out may contain an error record
            for rec in vars_out:
                if "error" in rec:
                    errors.append(rec)

    return tables, vars_all, errors


def main():
    parser = argparse.ArgumentParser(
        description="Scan Excel files recursively and report table sizes, variable names, counts, and binary stats."
    )
    parser.add_argument(
        "--root",
        type=str,
        default=r"C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah",
        help="Root folder to scan (recursively) for Excel files.",
    )
    parser.add_argument(
        "--out",
        type=str,
        default="excel_tables_report.xlsx",
        help="Output report file (.xlsx).",
    )
    parser.add_argument(
        "--also_csv",
        action="store_true",
        help="Also write CSV versions next to the XLSX report.",
    )
    args = parser.parse_args()

    root_dir = Path(args.root)
    out_path = Path(args.out)

    if not root_dir.exists():
        raise FileNotFoundError(f"Root folder does not exist: {root_dir}")

    excel_files = find_excel_files(root_dir)

    all_tables: List[dict] = []
    all_vars: List[dict] = []
    all_errors: List[dict] = []

    for fp in excel_files:
        tables, vars_out, errs = analyze_excel_file(fp)
        all_tables.extend(tables)
        all_vars.extend(vars_out)
        all_errors.extend(errs)

    tables_df = pd.DataFrame(all_tables).sort_values(["file", "sheet"], kind="stable") if all_tables else pd.DataFrame()
    vars_df = (
        pd.DataFrame(all_vars).sort_values(["file", "sheet", "variable"], kind="stable") if all_vars else pd.DataFrame()
    )
    errors_df = pd.DataFrame(all_errors) if all_errors else pd.DataFrame()

    # Write Excel report
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        (tables_df if not tables_df.empty else pd.DataFrame(columns=["file", "sheet", "rows", "cols", "columns"])).to_excel(
            writer, index=False, sheet_name="tables_summary"
        )
        (
            vars_df
            if not vars_df.empty
            else pd.DataFrame(
                columns=[
                    "file",
                    "sheet",
                    "variable",
                    "non_missing",
                    "missing",
                    "n_unique",
                    "is_binary",
                    "binary_0_count",
                    "binary_1_count",
                ]
            )
        ).to_excel(writer, index=False, sheet_name="variables_summary")
        (errors_df if not errors_df.empty else pd.DataFrame(columns=["file", "sheet", "error"])).to_excel(
            writer, index=False, sheet_name="errors"
        )

    if args.also_csv:
        base = out_path.with_suffix("")
        tables_df.to_csv(f"{base}_tables_summary.csv", index=False)
        vars_df.to_csv(f"{base}_variables_summary.csv", index=False)
        errors_df.to_csv(f"{base}_errors.csv", index=False)

    print(f"Scanned folder: {root_dir}")
    print(f"Excel files found: {len(excel_files)}")
    print(f"Tables (sheets) summarized: {len(all_tables)}")
    print(f"Variables summarized: {len(all_vars)}")
    print(f"Errors: {len(all_errors)}")
    print(f"Report written to: {out_path.resolve()}")


if __name__ == "__main__":
    main()
